In [ ]:
import os
os.environ["GROQ_API_KEY"] = "secret_key"

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Reply with exactly: OK"}
    ],
    temperature=0.5
)

print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

models = client.models.list()

for m in models.data:
    print(m.id)

In [ ]:
BASE_DIR = r"C:\Docs\Lectures\Spring_2026\Adv_Computer_Architecture\project\tasks"
OUTPUT_DIR = r"C:\Docs\Lectures\Spring_2026\Adv_Computer_Architecture\project\test_3_groq"
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
NUM_AGENTS = 84

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def clean_code(text):
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 3:
            text = parts[1]
            if "\n" in text:
                first_line, rest = text.split("\n", 1)
                if first_line.strip().lower() in {
                    "c", "cpp", "c++", "h", "hpp", "header"
                }:
                    text = rest
        text = text.strip()
    return text

def ask_agent(messages, user_content):
    messages.append({"role": "user", "content": user_content})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.5
    )
    reply = response.choices[0].message.content.strip()
    messages.append({"role": "assistant", "content": reply})
    return reply

# =========================
# SINGLE AGENT SETUP
# =========================
system_prompt = """You are participating in a step-by-step study session.

You will receive materials one stage at a time.
You must only work on the current stage.
Do not jump ahead.

Rules:
- When asked to answer a quiz, return only the quiz answers.
- When asked to complete a header file, return only the final completed header file.
- Do not include markdown code fences.
- Do not include explanations.
- Do not include extra text like 'Here is the solution'.
"""

# =========================
# STATIC INPUTS
# =========================
mem1_path = os.path.join(BASE_DIR, "mem_model_1.md")
mem1_text = read_text(mem1_path)

mem2_path = os.path.join(BASE_DIR, "mem_model_2.md")
mem2_text = read_text(mem2_path)

# =========================
# RUN 200 AGENTS
# =========================
for agent_num in range(1, NUM_AGENTS + 1):
    agent_id = f"{MODEL}_{agent_num:03d}"
    agent_output_dir = os.path.join(OUTPUT_DIR, agent_id)
    os.makedirs(agent_output_dir, exist_ok=True)

    print(f"Starting {agent_id} ...")

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    try:
        # =========================
        # STEP 1: MEMORY MODEL 1 QUIZ
        # =========================
        quiz1_prompt = f"""

Material:
{mem1_text}
"""
        quiz1_reply = ask_agent(messages, quiz1_prompt)
        save_text(os.path.join(agent_output_dir, "mem_model_1_quiz.txt"), quiz1_reply)

        # =========================
        # STEP 2: SPSC SC TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "spsc_sc_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "spsc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "spsc_sc_c", "spsc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 3: MPMC SC TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "mpmc_sc_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "mpmc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "mpmc_sc_c", "mpmc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 4: MEMORY MODEL 2 QUIZ
        # =========================
        quiz2_prompt = f"""

Material:
{mem2_text}
"""
        quiz2_reply = ask_agent(messages, quiz2_prompt)
        save_text(os.path.join(agent_output_dir, "mem_model_2_quiz.txt"), quiz2_reply)

        # =========================
        # STEP 5: SPSC RELAXED TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "spsc_relaxed_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "spsc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "spsc_relaxed_c", "spsc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 6: MPMC RELAXED TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "mpmc_relaxed_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "mpmc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "mpmc_relaxed_c", "mpmc_queue.h"),
            clean_code(task_reply)
        )

        print(f"Completed {agent_id}")

    except Exception as e:
        error_path = os.path.join(agent_output_dir, "error.txt")
        save_text(error_path, str(e))
        print(f"Failed {agent_id}: {e}")

print("All agent runs completed.")